# 06 — RQ4: Weighted-Scoring MCDA for Stage-1 Micro-Intervention Prioritization

**Question:** Which Stage-1 micro-intervention should be implemented first to achieve the largest controlled flow improvement?

**Method:** Weighted-scoring multi-criteria decision analysis (Saaty, 2008) over four candidate Stage-1 micro-interventions. Sensitivity analysis with ±10% weight perturbation across all six pairwise weight combinations confirms rank stability.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'

## Candidate Stage-1 micro-interventions

Each scored on three indicators (raw values are SME-elicited; replace with team estimates when running on real data):
1. **expected uplift in first-time-right** (proportion)
2. **expected reduction in SLA-breach rate** (proportion)
3. **implementation feasibility** (0–10 scale)

In [2]:
raw = pd.DataFrame({
    'intervention': ['Auto-completeness gating',
                     'Risk-score-based STP/escalation routing',
                     'Product-aware validation checklists',
                     'Channel-specific routing rules'],
    'ftr_uplift':            [0.082, 0.064, 0.041, 0.031],
    'sla_breach_reduction':  [0.058, 0.055, 0.030, 0.029],
    'feasibility':           [7.5,   8.0,   7.0,   9.0],
})
raw

,intervention,ftr_uplift,sla_breach_reduction,feasibility
0,Auto-completeness gating,0.082,0.058,7.5
1,Risk-score-based STP/escalation routing,0.064,0.055,8.0
2,Product-aware validation checklists,0.041,0.030,7.0
3,Channel-specific routing rules,0.031,0.029,9.0


## Min-max normalize each indicator to [0, 1]

In [3]:
norm = raw.copy()
for col in ['ftr_uplift', 'sla_breach_reduction', 'feasibility']:
    lo, hi = raw[col].min(), raw[col].max()
    norm[col + '_n'] = (raw[col] - lo) / (hi - lo)
norm[['intervention','ftr_uplift_n','sla_breach_reduction_n','feasibility_n']]

,intervention,ftr_uplift_n,sla_breach_reduction_n,feasibility_n
0,Auto-completeness gating,1.000000,1.000000,0.25
1,Risk-score-based STP/escalation routing,0.647059,0.896552,0.50
2,Product-aware validation checklists,0.196078,0.034483,0.00
3,Channel-specific routing rules,0.000000,0.000000,1.00


## Composite priority score (base case: 0.40 / 0.35 / 0.25)

In [4]:
weights_base = {'ftr_uplift_n': 0.40, 'sla_breach_reduction_n': 0.35,
                'feasibility_n': 0.25}
norm['score'] = sum(norm[c] * w for c, w in weights_base.items())
ranked = norm.sort_values('score', ascending=False)
ranked[['intervention', 'score']].round(3).reset_index(drop=True)

,intervention,score
0,Auto-completeness gating,0.812
1,Risk-score-based STP/escalation routing,0.698
2,Channel-specific routing rules,0.250
3,Product-aware validation checklists,0.091


## Sensitivity — ±10% perturbation across all weight pairs

In [5]:
import itertools
perturbations = []
weights = list(weights_base.items())
for (a, wa), (b, wb) in itertools.combinations(weights, 2):
    for delta in [+0.10, -0.10]:
        new = dict(weights_base)
        new[a] = wa * (1 + delta); new[b] = wb * (1 - delta)
        s = sum(new.values())
        for k in new: new[k] /= s
        scores = sum(norm[c] * w for c, w in new.items())
        top = norm.iloc[scores.idxmax()].intervention
        perturbations.append({'shifted_pair': f'{a}↔{b}',
            'delta': delta, 'top_intervention': top,
            'top_score': float(scores.max())})
pd.DataFrame(perturbations)

,shifted_pair,delta,top_intervention,top_score
0,ftr_uplift_n↔sla_breach_reduction_n,0.1,Auto-completeness gating,0.813433
1,ftr_uplift_n↔sla_breach_reduction_n,-0.1,Auto-completeness gating,0.811558
2,ftr_uplift_n↔feasibility_n,0.1,Auto-completeness gating,0.833744
3,ftr_uplift_n↔feasibility_n,-0.1,Auto-completeness gating,0.790609
4,sla_breach_reduction_n↔feasibility_n,0.1,Auto-completeness gating,0.832921
5,sla_breach_reduction_n↔feasibility_n,-0.1,Auto-completeness gating,0.791667


## Hypothesis test
H4₀ (equal priority across interventions) is rejected if the top score is materially higher than the second-place score AND the rank is stable under all ±10% perturbations.

In [6]:
scores = ranked.score.values
spread = scores[0] - scores[1]
top_stable = all(p['top_intervention'] == ranked.iloc[0].intervention
                 for p in perturbations)
print(f'Top score: {scores[0]:.3f}')
print(f'Spread to 2nd place: {spread:.3f}')
print(f'Top intervention stable under all perturbations: {top_stable}')
print()
print('H4_0 rejected:' if (spread > 0.05 and top_stable) else 'H4_0 not rejected:',
      ranked.iloc[0].intervention,
      'is the unambiguous first-tranche Stage-1 micro-intervention.')

Top score: 0.812
Spread to 2nd place: 0.115
Top intervention stable under all perturbations: True

H4_0 rejected: Auto-completeness gating is the unambiguous first-tranche Stage-1 micro-intervention.
